# NAMA : Adel Phonsa Jeanne V S

---
##NIM :25031554073


---

###Kelas: 2025G

1. Web Scraping | Key words : Blockchain Technology

In [ ]:
import json
import requests
import re
import time
import pandas as pd

from getpass import getpass
from IPython.display import display

In [ ]:
api_key = getpass("Masukkan API key Springer Nature: ")
endpoint = "https://api.springernature.com/meta/v2/json"
params = {
    "api_key": api_key,
    "q": "keyword:blockchain",
    "s": 1,
    "p": 5
}

response = requests.get(
    endpoint,
    params=params,
    timeout=60
)

print("HTTP status:", response.status_code)
response.raise_for_status()
data = response.json()
print("Informasi hasil:")
print(json.dumps(data.get("result", []), indent=2, ensure_ascii=False))
records = data.get("records", [])
if records:
    print("\nContoh record pertama:")
    print(json.dumps(records[0], indent=2, ensure_ascii=False))
else:
    print("Tidak ada record.")

In [ ]:

target = 50
page_size = 20
max_pages = 50

hasil = []
doi_tersimpan = set()

query = '"blockchain technology" AND type:Journal'
pola_topik = re.compile(
    r"\bblockchain[\s\u2010-\u2015-]+technology\b",
    re.I
)

for halaman in range(max_pages):
    if len(hasil) >= target:
        break

    posisi_awal = halaman * page_size + 1
    params = {
        "api_key": api_key,
        "q": query,
        "s": posisi_awal,
        "p": page_size
    }

    response = requests.get(
        endpoint,
        params=params,
        timeout=60
    )

    print("Halaman:", halaman + 1, "| HTTP:", response.status_code)
    if response.status_code != 200:
        print("Pengambilan dihentikan.")
        print(response.text)
        break

    data = response.json()
    records = data.get("records", [])

    if not records:
        print("Tidak ada data tambahan.")
        break

    for artikel in records:

        judul = str(artikel.get("title") or "").strip()
        abstrak = str(artikel.get("abstract") or "").strip()
        doi = str(artikel.get("doi") or "").strip()
        content_type = str(
            artikel.get("contentType") or ""
        ).lower().split()

        if "article" not in content_type:
            continue

        if str(artikel.get("language", "")).lower() != "en":
            continue

        if not abstrak or not doi:
            continue

        if doi.lower() in doi_tersimpan:
            continue

        if not pola_topik.search(judul + " " + abstrak):
            continue

        penulis = "; ".join(
            item.get("creator", "")
            for item in artikel.get("creators", [])
            if item.get("creator")
        )
        hasil.append({
            "judul": judul,
            "tanggal_terbit": artikel.get("publicationDate", ""),
            "authors": penulis,
            "abstrak": abstrak,
            "doi": doi,
            "doi_url": "https://doi.org/" + doi,
            "nama_jurnal": artikel.get("publicationName", ""),
            "content_type": artikel.get("contentType", ""),
            "bahasa": artikel.get("language", "")
        })
        doi_tersimpan.add(doi.lower())

        if len(hasil) >= target:
            break

    print("Artikel terkumpul:", len(hasil))

    time.sleep(1.2)
df = pd.DataFrame(hasil)
print("\nJumlah artikel:", len(df))
if len(df) >= target:
    df.to_csv(
        "artikel_blockchain_technology.csv",
        index=False,
        encoding="utf-8-sig"
    )

    print("CSV berhasil disimpan!")
elif len(df) > 0:
    print("Artikel terkumpul, tetapi belum mencapai 50.")
else:
    print("Belum ada artikel yang lolos penyaringan.")
display(df.head(10))


2.Preprocessing Kolom Abstrak

In [ ]:
!pip install nltk

In [ ]:
import nltk
import re

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

In [ ]:
def preprocessing(teks):

    # 1. Case Folding
    teks = teks.lower()

    # 2. Cleaning
    teks = re.sub(r'http\S+|www\S+', '', teks)
    teks = re.sub(r'[^a-z\s]', ' ', teks)

    # 3. Tokenization
    tokens = word_tokenize(teks)

    # 4. Stopword Removal
    tokens = [
        kata for kata in tokens
        if kata not in stop_words
    ]

    # 5. Stemming
    tokens = [
        stemmer.stem(kata)
        for kata in tokens
    ]

    return " ".join(tokens)

In [ ]:
df["abstrak_preprocessing"] = df["abstrak"].apply(preprocessing)
print("Preprocessing berhasil!")
display(
    df[["abstrak", "abstrak_preprocessing"]].head()
)

In [ ]:
df.to_csv(
    "artikel_blockchain_preprocessing.csv",
    index=False,
    encoding="utf-8-sig"
)
print("Dataset preprocessing berhasil disimpan!")

3. Analisis dan Insight Data Hasil Scraping

*1.  Analisis Jumlah Artikel Bersadarkan Tahun Terbit*

In [ ]:
import matplotlib.pyplot as plt

# Mengubah tanggal terbit menjadi format tanggal
df["tanggal_terbit"] = pd.to_datetime(df["tanggal_terbit"], errors="coerce")

# Mengambil tahun terbit
df["tahun"] = df["tanggal_terbit"].dt.year

# Menghitung jumlah artikel berdasarkan tahun
artikel_per_tahun = df["tahun"].value_counts().sort_index()
print("Jumlah artikel berdasarkan tahun terbit:")
display(artikel_per_tahun)

# Membuat grafik
artikel_per_tahun.plot(kind="bar", figsize=(10, 5))

plt.title("Jumlah Artikel Blockchain Technology Berdasarkan Tahun")
plt.xlabel("Tahun Terbit")
plt.ylabel("Jumlah Artikel")
plt.xticks(rotation=45)
plt.show()

*2. Penulis yang paling sering muncul*

In [ ]:
# Menggabungkan nama seluruh penulis
semua_penulis = df["authors"].dropna().str.split("; ").explode()

# Menghitung jumlah kemunculan penulis
jumlah_penulis = semua_penulis.value_counts().head(10)
print("10 penulis yang paling sering muncul:")
display(jumlah_penulis)

# Membuat grafik
jumlah_penulis.plot(kind="bar", figsize=(10, 5))

plt.title("10 Penulis Artikel Terbanyak")
plt.xlabel("Nama Penulis")
plt.ylabel("Jumlah Artikel")
plt.xticks(rotation=45, ha="right")
plt.show()

*3. Analisis panjang abstrak*

In [ ]:
# Menghitung jumlah kata pada setiap abstrak
df["jumlah_kata_abstrak"] = df["abstrak"].fillna("").apply(
    lambda x: len(x.split())
)
print("Statistik jumlah kata abstrak:")
display(df["jumlah_kata_abstrak"].describe())

# Membuat grafik
df["jumlah_kata_abstrak"].plot(kind="hist", bins=10, figsize=(10, 5))

plt.title("Distribusi Panjang Abstrak Artikel")
plt.xlabel("Jumlah Kata")
plt.ylabel("Jumlah Artikel")
plt.show()

*4. Analisis kata yang paling sering muncul*

In [ ]:
from collections import Counter

# Menggabungkan seluruh abstrak hasil preprocessing
semua_kata = " ".join(df["abstrak_preprocessing"].dropna())

# Memisahkan kata
kata = semua_kata.split()

# Menghitung frekuensi kata
frekuensi_kata = Counter(kata)

# Mengambil 10 kata paling sering muncul
kata_terbanyak = frekuensi_kata.most_common(10)
print("10 kata yang paling sering muncul:")
df_kata = pd.DataFrame(kata_terbanyak, columns=["Kata", "Frekuensi"])
display(df_kata)

# Membuat grafik
df_kata.plot(
    x="Kata",
    y="Frekuensi",
    kind="bar",
    figsize=(10, 5),
    legend=False
)

plt.title("10 Kata yang Paling Sering Muncul dalam Abstrak")
plt.xlabel("Kata")
plt.ylabel("Frekuensi")
plt.xticks(rotation=45)
plt.show()

4.  Feature Engineering

In [ ]:
# Mengubah hasil preprocessing menjadi kumpulan kata
kalimat = df["abstrak_preprocessing"].fillna("").apply(
    lambda x: x.split()
).tolist()

*1. One-Hot Encoding*

In [ ]:
from sklearn.preprocessing import MultiLabelBinarizer

# Membuat model One-Hot Encoding
onehot = MultiLabelBinarizer()
hasil_onehot = onehot.fit_transform(kalimat)
df_onehot = pd.DataFrame(
    hasil_onehot,
    columns=["onehot_" + kata for kata in onehot.classes_]
)
display(df_onehot.head())

*2. Bag of Words (BoW)*

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
bow = CountVectorizer(max_features=100)
hasil_bow = bow.fit_transform(
    df["abstrak_preprocessing"].fillna("")
)
df_bow = pd.DataFrame(
    hasil_bow.toarray(),
    columns=["bow_" + kata for kata in bow.get_feature_names_out()]
)
display(df_bow.head())

*3. TF-IDF*

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer(max_features=100)
hasil_tfidf = tfidf.fit_transform(
    df["abstrak_preprocessing"].fillna("")
)
df_tfidf = pd.DataFrame(
    hasil_tfidf.toarray(),
    columns=["tfidf_" + kata for kata in tfidf.get_feature_names_out()]
)
display(df_tfidf.head())

*4. Word2Vec*

In [ ]:
import numpy as np
from gensim.models import Word2Vec
model_w2v = Word2Vec(
    sentences=kalimat,
    vector_size=100,
    window=5,
    min_count=1,
    workers=4,
    sg=1
)
hasil_w2v = []
for kata in kalimat:
    vektor = [model_w2v.wv[k] for k in kata]
    if len(vektor) > 0:
        hasil_w2v.append(np.mean(vektor, axis=0))
    else:
        hasil_w2v.append(np.zeros(100))
df_w2v = pd.DataFrame(
    hasil_w2v,
    columns=["word2vec_" + str(i) for i in range(100)]
)
display(df_w2v.head())

In [ ]:
# Visualisasi Word2Vec
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

# Mengambil seluruh kata dan vektornya
kata = list(model_w2v.wv.index_to_key)
vektor = [model_w2v.wv[k] for k in kata]

# Mengubah vektor menjadi 2 dimensi
pca = PCA(n_components=2)
hasil_pca = pca.fit_transform(vektor)

# Membuat visualisasi
plt.figure(figsize=(15, 10))

plt.scatter(
    hasil_pca[:, 0],
    hasil_pca[:, 1]
)

# Menampilkan label seluruh kata
for i, k in enumerate(kata):
    plt.annotate(
        k,
        (hasil_pca[i, 0], hasil_pca[i, 1]),
        fontsize=8
    )
plt.title("Visualisasi Word2Vec")
plt.xlabel("Komponen 1")
plt.ylabel("Komponen 2")
plt.grid()
plt.show()

*5. FastText*

In [ ]:
from gensim.models import FastText
model_fasttext = FastText(
    sentences=kalimat,
    vector_size=100,
    window=5,
    min_count=1,
    workers=4
)
hasil_fasttext = []
for kata in kalimat:
    vektor = [model_fasttext.wv[k] for k in kata]
    if len(vektor) > 0:
        hasil_fasttext.append(np.mean(vektor, axis=0))
    else:
        hasil_fasttext.append(np.zeros(100))
df_fasttext = pd.DataFrame(
    hasil_fasttext,
    columns=["fasttext_" + str(i) for i in range(100)]
)
display(df_fasttext.head())

In [ ]:
# Menyimpan dataset ke csv
nama_file = "UTS_2025G_25031554073_Adel Phonsa Jeanne V S.csv"
df.to_csv(nama_file, index=False)
from google.colab import files
files.download(nama_file)